# Fuente 4 - API de Meteorología Histórica (Open-Meteo)

## Objetivo

Consultar datos meteorológicos históricos de distintos sectores de la Avenida Las Palmas mediante la API de Open-Meteo, para disponer de registros horarios de precipitación, temperatura y humedad relativa que puedan relacionarse temporalmente con la información de siniestralidad.

## 1. Identificación de la fuente

**Nombre:** Historical Weather API  
**Entidad:** Open-Meteo  
**Tipo de fuente:** API REST de información meteorológica histórica  
**Formato de respuesta:** JSON  
**URL:** https://open-meteo.com/en/docs/historical-weather-api  
**Endpoint:** https://archive-api.open-meteo.com/v1/archive  
**Cobertura:** tres puntos de referencia distribuidos a lo largo del corredor de la Avenida Las Palmas dentro de Medellín.

La fuente entrega estimaciones meteorológicas horarias mediante modelos y reanálisis. Las consultas espaciales representan sectores de referencia y no estaciones meteorológicas instaladas en cada curva. La asignación posterior de siniestros a la referencia meteorológica correspondiente requerirá un criterio espacial explícito.


In [ ]:
import requests
import pandas as pd
from IPython.display import display

url_api = "https://archive-api.open-meteo.com/v1/archive"

# Referencias espaciales de la Avenida Las Palmas, no sensores físicos
puntos_meteorologicos = [
    {"id_punto_clima": "CL-01", "latitud_referencia": 6.220055, "longitud_referencia": -75.562449},
    {"id_punto_clima": "CL-02", "latitud_referencia": 6.210226, "longitud_referencia": -75.545870},
    {"id_punto_clima": "CL-03", "latitud_referencia": 6.179181, "longitud_referencia": -75.543121},
]

fecha_inicio = "2019-01-01"
fecha_fin = "2025-12-31"

respuestas_clima = []
for punto in puntos_meteorologicos:
    parametros = {
        "latitude": punto["latitud_referencia"],
        "longitude": punto["longitud_referencia"],
        "start_date": fecha_inicio,
        "end_date": fecha_fin,
        "hourly": "precipitation,temperature_2m,relative_humidity_2m",
        "timezone": "America/Bogota"
    }
    respuesta = requests.get(url_api, params=parametros, timeout=180)
    respuesta.raise_for_status()
    datos = respuesta.json()
    if "error" in datos or "hourly" not in datos:
        raise ValueError(f"Respuesta meteorológica no válida en {punto['id_punto_clima']}: {datos}")
    respuestas_clima.append({"punto": punto, "datos": datos})
    print("Punto:", punto["id_punto_clima"], "- Registros:", len(datos["hourly"]["time"]))

print("Periodo:", fecha_inicio, "a", fecha_fin)
print("Zona horaria: America/Bogota")


## 2. Variables disponibles en la fuente

A partir de la respuesta de la API se identifican los campos horarios y sus unidades, para conocer la estructura de los registros descargados.

In [ ]:
unidades = respuestas_clima[0]["datos"].get("hourly_units", {})
variables = pd.DataFrame([
    {"nombre_variable": nombre, "unidad": unidades.get(nombre, "")}
    for nombre in respuestas_clima[0]["datos"]["hourly"].keys()
])
display(variables)


### Interpretación

Las variables seleccionadas incluyen:

- **Temporal:** `time`, fecha y hora local de cada registro.
- **Precipitación:** `precipitation`, acumulado de precipitación de la hora precedente, expresado en milímetros.
- **Temperatura:** `temperature_2m`, temperatura del aire a dos metros de altura, en grados Celsius.
- **Humedad:** `relative_humidity_2m`, humedad relativa del aire, en porcentaje.

Se utiliza la zona horaria `America/Bogota` para mantener coherencia con las horas de los registros de siniestralidad.

## 3. Consulta de registros de la fuente

Se consultan los datos horarios del periodo seleccionado y se revisan algunos de los registros devueltos.

In [ ]:
columnas_esperadas = ["time", "precipitation", "temperature_2m", "relative_humidity_2m"]
partes = []
for respuesta_clima in respuestas_clima:
    registros_punto = pd.DataFrame(respuesta_clima["datos"]["hourly"])
    faltantes = set(columnas_esperadas) - set(registros_punto.columns)
    if faltantes:
        raise ValueError(f"Faltan variables: {faltantes}")
    punto = respuesta_clima["punto"]
    registros_punto["id_punto_clima"] = punto["id_punto_clima"]
    registros_punto["latitud_referencia"] = punto["latitud_referencia"]
    registros_punto["longitud_referencia"] = punto["longitud_referencia"]
    partes.append(registros_punto)

registros = pd.concat(partes, ignore_index=True)
print("Cantidad de registros:", len(registros))
print("Columnas:", registros.columns.tolist())
display(registros.head(10))


## 4. Adquisición de los datos de la fuente

Los registros horarios obtenidos se conservan con los campos originales de la respuesta, sin reemplazar datos faltantes ni modificar sus valores. El archivo descargado servirá como entrada para las etapas posteriores de integración.

In [ ]:
df_meteorologia_raw = registros.copy()
print("Dimensiones de los datos adquiridos:", df_meteorologia_raw.shape)
display(df_meteorologia_raw.tail())

### Almacenamiento de la fuente adquirida

Los datos se almacenan en CSV, utilizando el mismo separador y codificación definidos para las otras fuentes del proyecto.

In [ ]:
nombre_archivo = "meteorologia_las_palmas.csv"

df_meteorologia_raw.to_csv(
    nombre_archivo,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Archivo generado:", nombre_archivo)
print("Registros almacenados:", len(df_meteorologia_raw))

### DESCARGAR

In [ ]:
try:
    from google.colab import files
    files.download(nombre_archivo)
except ImportError:
    print("Archivo guardado localmente:", nombre_archivo)

## 5. Calidad de los datos

Se revisan registros faltantes, duplicados, continuidad de la serie horaria y consistencia de las variables meteorológicas. Los valores de precipitación iguales a cero se consideran registros válidos; no deben confundirse con datos ausentes.

In [ ]:
df = pd.read_csv("meteorologia_las_palmas.csv", sep=";")
df["time"] = pd.to_datetime(df["time"], errors="coerce")

print("Registros analizados:", len(df))
print("Variables analizadas:", len(df.columns))
print("Datos faltantes por columna:")
display(df.isna().sum().rename("faltantes").reset_index(names="variable"))
print("Fechas inválidas:", df["time"].isna().sum())
print("Punto y hora duplicados:", df.duplicated(["id_punto_clima", "time"]).sum())

for identificador, grupo in df.groupby("id_punto_clima"):
    fechas_validas = grupo["time"].dropna().sort_values()
    if not fechas_validas.empty:
        horas_esperadas = pd.date_range(fechas_validas.min(), fechas_validas.max(), freq="h")
        horas_faltantes = horas_esperadas.difference(pd.DatetimeIndex(fechas_validas))
        print("Horas faltantes en", identificador, ":", len(horas_faltantes))

precipitacion = pd.to_numeric(df["precipitation"], errors="coerce")
temperatura = pd.to_numeric(df["temperature_2m"], errors="coerce")
humedad = pd.to_numeric(df["relative_humidity_2m"], errors="coerce")
print("Precipitación negativa:", (precipitacion < 0).sum())
print("Humedad fuera de 0 a 100 %:", ((humedad < 0) | (humedad > 100)).sum())
print("Valores de temperatura no numéricos:", temperatura.isna().sum())


## 6. Distribución de horas con y sin lluvia

Se considera **hora con lluvia** aquella cuyo registro horario de precipitación es mayor que cero. Los valores faltantes se mantienen como **Sin dato**, sin clasificarlos como horas secas.

In [ ]:
precipitacion = pd.to_numeric(df["precipitation"], errors="coerce")
clasificacion = pd.Series("Sin dato", index=df.index, dtype="string")
clasificacion.loc[precipitacion.notna() & (precipitacion == 0)] = "Sin lluvia"
clasificacion.loc[precipitacion.notna() & (precipitacion > 0)] = "Con lluvia"

resumen = (
    df.assign(condicion=clasificacion)
    .groupby(["id_punto_clima", "condicion"], dropna=False)
    .size()
    .reset_index(name="horas")
)
display(resumen)

print("Precipitación total estimada (mm) por punto:")
display(df.assign(precipitacion=precipitacion).groupby("id_punto_clima")["precipitacion"].sum())
print("Horas con datos válidos:", precipitacion.notna().sum())


## 7. Contribución de la fuente al problema

La fuente meteorológica aporta registros históricos horarios para diferentes referencias espaciales del corredor de Las Palmas. Estos datos podrán asociarse a los siniestros según la ubicación y fecha de cada evento para identificar condiciones de lluvia o ausencia de lluvia.

La comparación proporcional de accidentes debe considerar las horas lluviosas y secas disponibles para cada referencia meteorológica. Las estimaciones espaciales de la API no equivalen a mediciones exactas de lluvia en cada curva, y se validará la compatibilidad de fechas y horas en la integración posterior.
